## CUSTOMER CHURN PREDICTION ESSAY
####  CARRIED OUT BY:
#### OMOTOSO FIYINFOLUWA VICTORIA - victoriaomotoso0303@gmail.com
#### SHODIJO HEPHZIBAH OLAMIDE - shodijohephzibah2005@gmail.com

This notebook provides a plain-language walkthrough of the full project. It explains what was done at each major step, why it was done, and what the results mean. It is meant to be read alongside the main notebook rather than run independently.

## OVERVIEW

Customer churn is one of the more pressing problems banks deal with. When customers leave, it hits revenue hard, and the tricky part is that most institutions only notice after the fact. This project takes a proactive approach by using machine learning to predict which customers are likely to churn before they actually do.

The dataset contains 10,000 bank customer records with features including credit score, age, account balance, number of products, geography, gender, and whether the customer is an active member. The target variable, Exited, tells us whether a customer stayed or left.

Three models are built and compared: Logistic Regression as the baseline, a Decision Tree Regressor to capture nonlinear patterns, and a Random Forest Regressor as the strongest ensemble approach.

## STEP 1 — DATA LOADING AND INITIAL INSPECTION

The dataset is loaded from a CSV file and given an initial inspection using df.shape, df.head(), df.info(), and df.describe().

**What this tells us:**
- The dataset has 10,000 rows and 14 columns
- 3 columns (RowNumber, CustomerId, Surname) are identifiers that carry no predictive value and are dropped early
- All columns have the correct data types with no obvious issues
- The average customer is around 39 years old, has been with the bank for about 5 years, and holds roughly 1.5 products

## STEP 2 — DATA QUALITY CHECKS

Two checks are run: one for missing values and one for zero values.

**Missing values:** None found across all columns, so no imputation is needed.

**Zero values:** Zeros appear in Tenure, Balance, HasCrCard, IsActiveMember, and Exited. All of these are valid. A customer can genuinely have a zero balance, no credit card, or be a brand new account holder. No data cleaning is needed here.

## STEP 3 — TARGET VARIABLE EXPLORATION

The Exited column is visualized to understand the class distribution.

**Finding:** About 80% of customers stayed and 20% churned. This 80/20 split means the dataset is imbalanced, which is important because a model can achieve 80% accuracy by simply predicting everyone stays. This has to be accounted for during modeling.

## STEP 4 — DISTRIBUTION OF NUMERICAL FEATURES

Histograms with KDE curves are plotted for CreditScore, Age, Tenure, Balance, NumOfProducts, and EstimatedSalary.

**Key findings:**
- CreditScore is roughly symmetric with no major skew
- Age skews slightly right, most customers are middle aged
- Tenure is evenly spread across 0 to 10 years
- Balance has a bimodal distribution, a large spike at zero and a cluster of higher value accounts
- NumOfProducts is concentrated at 1 and 2, very few customers hold more
- EstimatedSalary is uniformly distributed, possibly synthetic

## STEP 5 — NUMERICAL FEATURES AGAINST TARGET

Boxplots compare the distribution of each numerical feature between customers who stayed (0) and those who churned (1).

**Key findings:**
- Churners tend to be older on average
- Churners tend to have higher account balances
- CreditScore and Tenure show minimal difference between the two groups
- NumOfProducts shows a notable pattern, customers with 3 or more products churn at a much higher rate

## STEP 6 — CATEGORICAL FEATURES ANALYSIS

Count plots show the distribution of Geography, Gender, HasCrCard, and IsActiveMember, both on their own and broken down by churn status.

**Key findings:**
- Most customers are from France, but Germany has a disproportionately high churn rate
- Gender shows almost no difference in churn rates
- Credit card ownership has a small effect on churn
- Inactive members churn at a significantly higher rate than active ones, making IsActiveMember one of the strongest early signals

## STEP 7 — CORRELATION MATRIX

A heatmap of correlations between all numerical features is generated.

**Key findings:**
- Most features are weakly correlated with each other, which is good for modeling
- Age has the strongest positive correlation with Exited at 0.29
- IsActiveMember has a negative correlation with Exited at -0.16
- Balance and NumOfProducts have a moderate negative correlation of -0.3 with each other, but this is not strong enough to cause serious multicollinearity issues

## STEP 8 — FEATURE ENGINEERING AND PREPROCESSING

Before modeling, two preprocessing steps are applied:

**One-Hot Encoding:** Gender and Geography are categorical text columns. Machine learning models require numerical input, so these are converted into binary columns using OneHotEncoder. Gender becomes Gender_Female and Gender_Male. Geography becomes Geography_France, Geography_Germany, and Geography_Spain.

**Train-Test Split:** The data is split 80/20 into training and test sets. Stratification is used to ensure both sets maintain the same 80/20 churn ratio, which is important given the class imbalance.

## STEP 9 — LOGISTIC REGRESSION

Logistic Regression is used as the baseline model. It is straightforward, interpretable, and well suited for binary classification.

**Key decision:** class_weight='balanced' is set to compensate for the class imbalance. Without this, the model would default to predicting the majority class most of the time and miss nearly all churners.

**Results:**
- Training Accuracy: 71.03%
- Test Accuracy: 71.45%
- Confusion Matrix: [[1142, 451], [120, 287]]

The accuracy drop compared to an unbalanced model is expected and acceptable. The model correctly identified 287 churners with only 120 missed, which is a much more useful result for a churn prediction task. Feature coefficients show that Geography_Germany and Age push toward churn, while IsActiveMember is the strongest negative coefficient.

## STEP 10 — DECISION TREE REGRESSOR

A Decision Tree Regressor is used to capture nonlinear relationships that logistic regression cannot. It outputs continuous probability values rather than clean 0s and 1s, so a threshold is applied to convert predictions to binary labels.

**Key decisions:**
- max_depth=10, min_samples_split=20, min_samples_leaf=10 are set to prevent overfitting
- A threshold of 0.4 is used instead of the default 0.5 to make the model more sensitive to churners

**Results:**
- Training Accuracy: 87.93%
- Test Accuracy: 82.95%
- Confusion Matrix: [[1440, 153], [188, 219]]

The Decision Tree catches 219 churners and misses 188, a clear improvement over logistic regression in overall accuracy. The tree visualization shows Age and NumOfProducts as the first splitting features, which aligns with what EDA revealed.

## STEP 11 — RANDOM FOREST REGRESSOR

The Random Forest builds 100 Decision Trees and averages their predictions, reducing the variance of any single tree and producing more stable results.

**Key decisions:**
- n_estimators=100 trees are used
- The same depth constraints as the Decision Tree are applied to each individual tree
- A threshold of 0.3 is chosen over 0.4 because it catches 275 churners compared to 225 at 0.4, with only 132 missed versus 182. In a churn prediction context, catching more at-risk customers matters more than raw accuracy.

**Results:**
- Training Accuracy: 88.15%
- Test Accuracy: 84.30%
- Confusion Matrix: [[1411, 182], [132, 275]]

The Random Forest is the strongest model in the project. Feature importance averaged across all 100 trees consistently identifies Age, NumOfProducts, Balance, and IsActiveMember as the top predictors of churn.

## CONCLUSION

This project set out to predict customer churn using three machine learning models: Logistic Regression, Decision Tree Regressor, and Random Forest Regressor. Each model brought something different to the table, and looking at them together tells a clearer story than any one model alone.

The Logistic Regression with class balancing achieved a test accuracy of 71.45% but was strong at catching churners, identifying 287 out of 407. The Decision Tree improved overall accuracy to 82.95% and with a threshold of 0.4 correctly identified 219 churners. The Random Forest was the strongest performer overall, catching 275 churners with only 132 missed at a test accuracy of 84.30%.

Across all three models, Age, NumOfProducts, Balance, and IsActiveMember consistently emerged as the most important predictors of churn. These findings align with the patterns observed during EDA, which gives confidence that the models are picking up on real signals rather than noise.

Overall, this project demonstrates that machine learning can meaningfully support customer retention efforts. With the right model and the right evaluation focus, banks can identify at-risk customers early and act before they leave.

#### PROJECT TEAM
##### EDA: AYEVBOSA AJAYI-EBOHON | SIYANBOLA OLUWATOBI ABIGAIL
##### MODELLING: OBA OLAITAN | TEMILOLUWA MAYOWA AWOYEMI | IBEKWE GHISLIAN CHIGOZIE